# Tubes 2 IF3270 â€” CNN, RNN & LSTM
**Dataset CNN**: Intel Image Classification (~25.000 gambar, 6 kelas)
**Dataset RNN/LSTM**: Flickr8k (image captioning)


In [ ]:
# =============================================================================
# GPU Setup â€” jalankan PERTAMA sebelum import TF lain
# =============================================================================
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'[GPU] Ditemukan {len(gpus)} GPU: {[g.name for g in gpus]}')
    print(f'[GPU] TensorFlow versi: {tf.__version__}')
else:
    print('[GPU] Tidak ada GPU terdeteksi â€” menggunakan CPU.')
    print('      Untuk GPU NVIDIA di Windows, install: pip install tensorflow==2.10.0')
    print('      Atau gunakan WSL dan: pip install tensorflow[and-cuda]')
    print(f'[TF] Versi TensorFlow: {tf.__version__}')

print(f'[TF] Built with CUDA: {tf.test.is_built_with_cuda()}')

In [ ]:
# ── Colab: Mount Google Drive (skip jika lokal) ────────────────────────────────
import sys

IN_COLAB = 'google.colab' in sys.modules or 'google.colab' in str(type(None))
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('[Colab] Google Drive terpasang.')
else:
    print('[Lokal] Tidak di Colab — skip Drive mount.')

In [ ]:
import os
import sys
import numpy as np

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    # Path Google Drive
    DRIVE_ROOT  = '/content/drive/MyDrive/tubes2'
    SRC_DIR     = os.path.join(DRIVE_ROOT, 'src')
    DATA_DIR    = os.path.join(DRIVE_ROOT, 'data', 'intel_image_classification')
    WEIGHTS_DIR = os.path.join(DRIVE_ROOT, 'weights', 'cnn')
    RESULTS_DIR = os.path.join(DRIVE_ROOT, 'results')
else:
    # Path lokal — notebook ada di src/
    SRC_DIR     = os.path.abspath('.')
    if not os.path.exists(os.path.join(SRC_DIR, 'cnn')):
        SRC_DIR = os.path.join(os.path.abspath('.'), 'src')
    PROJECT_ROOT = os.path.dirname(SRC_DIR)
    DATA_DIR    = os.path.join(PROJECT_ROOT, 'data', 'intel_image_classification')
    WEIGHTS_DIR = os.path.join(PROJECT_ROOT, 'weights', 'cnn')
    RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')

sys.path.insert(0, SRC_DIR)
sys.path.insert(0, os.path.join(SRC_DIR, 'shared'))

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'[Path] SRC_DIR    : {SRC_DIR}')
print(f'[Path] DATA_DIR   : {DATA_DIR}  (exists: {os.path.exists(DATA_DIR)})')
print(f'[Path] WEIGHTS_DIR: {WEIGHTS_DIR}')
print(f'[Path] RESULTS_DIR: {RESULTS_DIR}')

if IN_COLAB:
    assert os.path.exists(SRC_DIR),  f'src/ tidak ditemukan di Drive: {SRC_DIR}'
    assert os.path.exists(DATA_DIR), f'Dataset tidak ditemukan di Drive: {DATA_DIR}'
    seg_train = os.path.join(DATA_DIR, 'seg_train')
    seg_test  = os.path.join(DATA_DIR, 'seg_test')
    assert os.path.exists(seg_train), f'seg_train/ tidak ada: {seg_train}'
    assert os.path.exists(seg_test),  f'seg_test/ tidak ada: {seg_test}'
    print('[Path] Semua path Colab OK ✓')

## Bagian 1: Utility Functions (PIL/Pillow + NumPy)

In [ ]:
from shared.preprocessing import load_image, load_batch, extract_features

# Test load_image
# img = load_image('path/ke/gambar.jpg', target_size=(150, 150))
# print(f'Image shape: {img.shape}, dtype: {img.dtype}, range: [{img.min():.2f}, {img.max():.2f}]')

# Test load_batch
# batch = load_batch(['path1.jpg', 'path2.jpg'], target_size=(150, 150))
# print(f'Batch shape: {batch.shape}')  # (N, 150, 150, 3)

## Bagian 2: Forward Propagation From Scratch

In [ ]:
from cnn.scratch.conv2d import Conv2D
from cnn.scratch.locally_connected2d import LocallyConnected2D
from cnn.scratch.pooling import MaxPooling2D, AveragePooling2D, GlobalAveragePooling2D
from cnn.scratch.flatten import Flatten
from cnn.scratch.model_scratch import CNNScratch

## Bagian 3: Pelatihan Model (Keras)

Variasi hyperparameter (16 arsitektur):
- Jumlah layer konvolusi: [2, 4]
- Jumlah filter: [32, 128]
- Ukuran kernel: [(3,3), (5,5)]
- Pooling: ['max', 'average']

Total: 2 Ã— 2 Ã— 2 Ã— 2 = **16 arsitektur**

In [ ]:
from shared.intel_preprocess import IntelImagePreprocessor

preprocessor = IntelImagePreprocessor(DATA_DIR, target_size=(150, 150))
preprocessor.load_data()
preprocessor.summary()

In [ ]:
from cnn.keras.train import train_with_variations

results = train_with_variations(
    data_dir=DATA_DIR,
    arch_type='conv2d',
    layer_variations=[2, 4],
    filter_variations=[32, 128],
    kernel_variations=[(3, 3), (5, 5)],
    pooling_variations=['max', 'average'],
    epochs=30,
    batch_size=32,
    weights_dir=WEIGHTS_DIR,
    results_path=os.path.join(RESULTS_DIR, 'cnn_variations.json')
)

In [ ]:
# ── Ranking hasil variasi ──────────────────────────────────────────────────────
ranked = sorted(
    [(k, v) for k, v in results.items() if 'best_val_f1' in v],
    key=lambda x: x[1]['best_val_f1'],
    reverse=True
)

print(f'
{"="*60}')
print('  RANKING — Val Macro F1')
print(f'{"="*60}')
for i, (name, res) in enumerate(ranked, 1):
    print(f'  {i:2d}. {name:45s} F1={res["best_val_f1"]:.4f}')

if ranked:
    best_name = ranked[0][0]
    best_f1   = ranked[0][1]['best_val_f1']
    print(f'
  BEST : {best_name}  (F1={best_f1:.4f})')
    print(f'  Bobot: {WEIGHTS_DIR}/{best_name}.h5')

In [ ]:
# ── Verifikasi bobot tersimpan ─────────────────────────────────────────────────
saved = [f for f in os.listdir(WEIGHTS_DIR) if f.endswith('.h5')]
print(f'Bobot tersimpan ({len(saved)} file):')
for f in sorted(saved):
    size_mb = os.path.getsize(os.path.join(WEIGHTS_DIR, f)) / 1e6
    print(f'  {f}  ({size_mb:.1f} MB)')

## Bagian 4: Eksperimen dan Evaluasi

In [ ]:
from cnn.keras.evaluate import run_part4_evaluation

# Evaluasi dengan best model dari Bagian 3
# run_part4_evaluation(...)